# J09 — Transfert de chaleur dans le sol

**Physique et hydrodynamique des sols** — S. J. Gumiere

*Atelier du Jour 9 : propriétés thermiques, onde amortie, schéma implicite, module Heat Transport de HYDRUS-1D*

> Version **étudiant** : complétez les cellules marquées `# À COMPLÉTER`.

## Mise en place

Unités : SI pour les propriétés (W m$^{-1}$ K$^{-1}$, J m$^{-3}$ K$^{-1}$), puis cm et jours pour les simulations (HYDRUS : g, cm, j).
Les projets HYDRUS-1D sont dans `../../hydrus/` (lecteurs dans `../hydrus_io.py`).

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, special, linalg
from hydrus_io import read_obs_node, read_nod_inf, read_tlevel

plt.rcParams.update({"figure.figsize": (7.5, 4), "axes.grid": True, "grid.alpha": 0.3})
HYD = "../../hydrus"
OM = 2 * np.pi          # pulsation journalière (rad/j)
S2J = 86400.0           # secondes par jour

## Exercice 1 — Propriétés thermiques en fonction de la teneur en eau  *(≈ 12 min)*

Conductivité thermique de Chung & Horton (1987) : $\lambda(\theta) = b_1 + b_2\theta + b_3\theta^{0,5}$ (W m$^{-1}$ K$^{-1}$), avec
sable (0,228 ; −2,406 ; 4,909), loam (0,243 ; 0,393 ; 1,534), argile (−0,197 ; −0,962 ; 2,521) — valeurs par défaut de HYDRUS-1D.
Capacité thermique volumique de de Vries : $C = \theta_n C_n + \theta_o C_o + \theta C_w$ avec $C_n = 1{,}92$, $C_o = 2{,}51$,
$C_w = 4{,}18$ MJ m$^{-3}$ K$^{-1}$ ; $\theta_n = 1 - \theta_s - \theta_o$ ($\theta_s$ = 0,43 / 0,43 / 0,38 ; $\theta_o = 0$).

1. Écrire `lam_CH(theta, sol)` et `C_vol(theta, sol)` ; tracer $\lambda(\theta)$, $C(\theta)$ et $D_T = \lambda/C$ (en cm²/j) pour les trois sols.
2. À $\theta = 0{,}6\,\theta_s$, calculer $D_T$, la profondeur d'amortissement journalière $d = \sqrt{2D_T/\omega}$ et annuelle.
3. Convertir $b_1, b_2, b_3$ (W m$^{-1}$ K$^{-1}$ → g cm j$^{-3}$ K$^{-1}$) et $C_n, C_o, C_w$ (J m$^{-3}$ K$^{-1}$ → g cm$^{-1}$ j$^{-2}$ K$^{-1}$)
   pour le loam. Démontrer les facteurs (1 W = 1 kg m² s$^{-3}$) puis comparer aux valeurs du bloc E de `J09_chaleur_sans_convection/SELECTOR.IN`
   (les lire dans le fichier).
4. Pour le loam à $h = -100$ cm ($\theta = 0{,}243$, projet HYDRUS), donner $\lambda$, $C$, $D_T$ et $d$ : ce sont les valeurs de référence de l'exercice 4.

In [ ]:
CH = {"sable": (0.228, -2.406, 4.909), "loam": (0.243, 0.393, 1.534), "argile": (-0.197, -0.962, 2.521)}
THS = {"sable": 0.43, "loam": 0.43, "argile": 0.38}
Cn, Co, Cw = 1.92e6, 2.51e6, 4.18e6      # J m-3 K-1

def lam_CH(theta, sol):
    # À COMPLÉTER
    pass

def C_vol(theta, sol, theta_o=0.0):
    # À COMPLÉTER
    pass

# 1. tracés lambda, C, D_T (cm²/j : D_T[m²/s] * 1e4 * 86400)
# 2. d journalière / annuelle à theta = 0.6 theta_s
# 3. facteurs de conversion et lecture de SELECTOR.IN (bloc E : ligne après "thn tho lambda b1 ...")
# 4. loam à theta = 0.243

## Exercice 2 — Ajustement d'une onde amortie sur des températures mesurées  *(≈ 15 min)*

`data/J09_temperatures_horaires.csv` : températures horaires « mesurées » à 5, 10, 20 et 40 cm pendant 5 jours (`t_j`, `T_5cm`, … ; bruit
de capteur et dérive lente de la moyenne).

1. Tracer les quatre séries. Pour chaque profondeur, ajuster par `curve_fit` le modèle $T = T_m + a\,t + A\sin(\omega t + \varphi)$
   ($\omega = 2\pi$ rad/j connu) ; récupérer $A(z) > 0$ et $\varphi(z)$ (ramener $A$ positif et $\varphi$ continu avec `np.unwrap`).
2. Régresser $\ln A$ sur $z$ (pente $-1/d_A$) et $\varphi$ sur $z$ (pente $-1/d_\varphi$) : deux estimations de $d$, donc de
   $D_T = \omega d^2/2$. Comparer entre elles et à la valeur du loam (exercice 1).
3. Quelle est l'heure du maximum à chaque profondeur ? Vérifier que le retard croît linéairement avec $z$.
4. Refaire l'ajustement en omettant la dérive ($a = 0$) : effet sur $A$, $\varphi$ et $d$ ? Pourquoi la profondeur 40 cm est-elle la plus incertaine ?

In [ ]:
tp = pd.read_csv("data/J09_temperatures_horaires.csv")
t = tp["t_j"].to_numpy(); zs = np.array([5, 10, 20, 40])

def modele(t, Tm, a, A, phi):
    return Tm + a * t + A * np.sin(OM * t + phi)

# 1. ajustement par profondeur (p0 = [20, 0, 5, 0]) ; A > 0 ; phi unwrap
# À COMPLÉTER

# 2. régressions ln A vs z, phi vs z (np.polyfit) -> d_A, d_phi, D_T
# À COMPLÉTER

# 3. heure du maximum : OM t + phi = pi/2 -> t_max = (pi/2 - phi)/OM (modulo 1 j)
# À COMPLÉTER

# 4. sans dérive
# À COMPLÉTER

## Exercice 3 — Schéma implicite pour l'équation de la chaleur  *(≈ 18 min)*

Résoudre $\partial T/\partial t = D_T\,\partial^2 T/\partial z^2$ sur une colonne de 100 cm avec $T(0,t) = T_m + A_0\sin(\omega t)$
et gradient nul au bas, par le schéma $\vartheta$ : $T_i^{n+1} - T_i^n = r[\vartheta\,\delta^2T_i^{n+1} + (1-\vartheta)\,\delta^2T_i^n]$,
$r = D_T\Delta t/\Delta z^2$ ($\vartheta$ = 0 explicite, 1 implicite, 0,5 Crank–Nicolson).

1. Écrire `chaleur_1d(L, nz, dt, tmax, DT, theta_s, Tm=20, A0=10, T_init=None)` (matrice tridiagonale, `scipy.linalg.solve_banded`,
   nœud fantôme pour le gradient nul) qui renvoie `z`, `T(z, tmax)` et la série `T(10 cm, t)`. Vectoriser (pas de boucle sur les nœuds).
2. Vérification : partir de la solution analytique à $t = 0$ ; erreur RMS sur 0–50 cm à $t = 2{,}25$ j en fonction de $\Delta t$
   (implicite et CN, $\Delta z$ = 0,25 cm) et de $\Delta z$ (CN, $\Delta t$ = 0,005 j). Retrouver les ordres 1 et 2.
3. Bilan d'énergie : vérifier que la variation d'énergie stockée $C\int (T - T_{init})\,dz$ égale l'intégrale du flux entrant en surface
   $-\lambda\,\partial T/\partial z|_0$ (différence décentrée d'ordre 2) sur une simulation implicite de 2 j ($C$ et $\lambda$ du loam, exercice 1).
4. Explicite : montrer que la solution diverge pour $r > 1/2$ ($\Delta z$ = 2 cm, $\Delta t = 1{,}03\,\Delta z^2/2D_T$) et reste bornée pour $r = 0{,}45$.

In [ ]:
def chaleur_1d(L, nz, dt, tmax, DT, theta_s, Tm=20.0, A0=10.0, T_init=None, z_obs=10.0):
    z = np.linspace(0, L, nz); dz = z[1] - z[0]; r = DT * dt / dz ** 2
    T = np.full(nz, Tm) if T_init is None else T_init(z)
    # matrice bande (3, nz) : diagonale 1 + 2 theta_s r, hors-diagonales -theta_s r ; ligne 0 = identité (Dirichlet) ;
    # dernier noeud : coefficient sous-diagonal doublé (noeud fantôme, gradient nul)
    # À COMPLÉTER
    pass

def analytique(z, t, DT, Tm=20.0, A0=10.0):
    d = np.sqrt(2 * DT / OM)
    return Tm + A0 * np.exp(-z / d) * np.sin(OM * t - z / d)

# 2. erreurs vs dt et dz
# 3. bilan d'énergie
# 4. explicite instable

## Exercice 4 — HYDRUS-1D : conduction seule et conduction + convection  *(≈ 15 min)*

Projets `J09_chaleur_sans_convection` (loam, $h = -100$ cm, flux d'eau nul) et `J09_chaleur_avec_convection` (infiltration permanente
5 cm/j, $\theta = 0{,}405$) : surface à $20 \pm 10$ °C (période 1 j, maximum à 13 h), nœuds d'observation 5, 10, 20, 40 cm, 5 jours.

1. Lire `OBS_NODE.OUT` des deux projets (`read_obs_node`, variable `Temp`, nœuds 6, 11, 21, 41) et tracer $T(t)$ aux quatre profondeurs.
2. Sur le dernier jour ($t \ge 4$), ajuster $T = T_m + A\sin(\omega t + \varphi)$ à chaque profondeur ; régresser $\ln A$ et $\varphi$ sur $z$ :
   $d$ (amortissement) et longueur de phase, pour les deux projets.
3. Sans convection : comparer $d$ à $\sqrt{2D_T/\omega}$ avec $D_T$ du loam à $\theta = 0{,}243$ (exercice 1) et l'amplitude à chaque
   profondeur à $A_0 e^{-z/d}$.
4. Avec convection : calculer le nombre d'onde complexe $k = [-C_w q + \sqrt{(C_w q)^2 + 4 i\omega C\lambda}]/(2\lambda)$ (unités cm, j, g :
   $\lambda = \lambda_0(0{,}405) + \beta_T C_w q$, $\beta_T = 5$ cm, $q = 5$ cm/j) ; comparer $1/\mathrm{Re}(k)$ et $1/\mathrm{Im}(k)$ aux valeurs
   ajustées. Quelle part de la pénétration accrue vient de la convection, de la dispersion, du changement de $\theta$ ?

In [ ]:
obs = {"sans": read_obs_node(f"{HYD}/J09_chaleur_sans_convection"), "avec": read_obs_node(f"{HYD}/J09_chaleur_avec_convection")}
noeuds = {6: 5, 11: 10, 21: 20, 41: 40}

# 1. tracés
# 2. ajustement sinusoïdal sur t >= 4 (curve_fit, p0 = [20, 5, 0]) ; A > 0 ; unwrap ; régressions
def amp_phase(df, nd, t0=4.0):
    # À COMPLÉTER
    pass

# 3. comparaison à d = sqrt(2 D_T / OM), A0 exp(-z/d)
# 4. nombre d'onde complexe (unités HYDRUS : lambda en g cm j^-3 K^-1, C en g cm^-1 j^-2 K^-1, q en cm/j)
f_lam = 1e3 * 1e2 * S2J ** 3; f_C = 1e3 * 1e-2 * S2J ** 2
# À COMPLÉTER

## Exercice 5 — Bonus — flux de chaleur en surface et énergie stockée  *(≈ facultatif)*

À partir des profils `NOD_INF.OUT` du projet sans convection (toutes les 3 h), calculer le flux de chaleur en surface
$G(0,t) = -\lambda\,\partial T/\partial z|_0$ (W/m², différence décentrée d'ordre 2, $\lambda = 1{,}095$ W m$^{-1}$ K$^{-1}$) et le comparer à
$G = \sqrt2\lambda A_0/d\,\sin(\omega t - 7\pi/12 + \pi/4)$. En déduire l'énergie stockée entre le minimum et le maximum de $T_s$ (MJ/m²).

In [ ]:
nod = read_nod_inf(f"{HYD}/J09_chaleur_sans_convection")
# À COMPLÉTER : pour chaque temps, G = -lam * (-3 T0 + 4 T1 - T2) / (2 dz) avec dz en m

## Pour aller plus loin

* Onde annuelle + onde journalière superposées : simuler 2 ans avec le schéma implicite et retrouver le retard saisonnier à 1 m.
* Coupler $\lambda(\theta(z,t))$ et $C(\theta(z,t))$ au profil d'humidité du J8 (redistribution) dans le solveur de l'exercice 3.
* Estimer $D_T$ par HYDRUS inverse (J10) à partir des séries de l'exercice 2.